# LIMBA 2.0 — Training Notebook

This notebook documents the parameter-efficient fine-tuning of Meta Llama 3.1 8B for Italian and Limba Sarda Comuna (LSC).

The workflow uses Unsloth and LoRA, trains on an instruction–response dataset, validates the model before and after training, and exports the result to GGUF Q4_K_M format.

> **Security note:** authentication tokens are never stored directly in this notebook.


In [ ]:
# Installazione silenziosa e forzata delle dipendenze
# -q: quiet mode (nasconde i messaggi di download)
# --no-warn-conflicts: ignora i messaggi rossi sulle dipendenze secondarie
print("Inizializzazione ambiente in corso... attendere circa 1 minuto.")

!pip install --upgrade --no-cache-dir -q unsloth
!pip install --no-deps -q xformers trl peft accelerate bitsandbytes

print("Installazione completata con successo.")

In [ ]:
from unsloth import FastLanguageModel
import torch

# Impostiamo la finestra di contesto a 4096 per gestire risposte ampie
max_seq_length = 4096
dtype = None
load_in_4bit = True # Riduce l'occupazione della VRAM a circa 5GB

# Carichiamo il modello base e il tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Configurazione degli adattatori LoRA per le 1500 righe
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# Esecuzione del test pre-training (Domanda 1)

FastLanguageModel.for_inference(model)

domanda_1 = "Chi sei e chi ti ha creato?"
prompt_1 = f"""### Istruzione:
{domanda_1}

### Risposta:"""

inputs = tokenizer([prompt_1], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    temperature = 0.1,
    repetition_penalty = 1.1
)

print("\n--- TEST PRE-TRAINING (RE-RUN): DOMANDA 1 ---")
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Esecuzione del test pre-training (Domanda 2)

FastLanguageModel.for_inference(model)

domanda_2 = "Chi è Grazia Deledda?"

# Usiamo un formato che simula quello che il modello vedrà nel dataset
prompt_2 = f"""### Istruzione:
{domanda_2}

### Risposta:"""

inputs = tokenizer([prompt_2], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    temperature = 0.1,
    repetition_penalty = 1.2
)

print("\n--- TEST PRE-TRAINING: DOMANDA 2 ---")
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Esecuzione del test pre-training (Domanda 3):

FastLanguageModel.for_inference(model)

domanda_3 = "Cosa è la fotosintesi?"

prompt_3 = f"""### Istruzione:
{domanda_3}

### Risposta:"""

inputs = tokenizer([prompt_3], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    temperature = 0.1,
    repetition_penalty = 1.2
)

print("\n--- TEST PRE-TRAINING: DOMANDA 3 ---")
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
from datasets import load_dataset

# Caricamento del dataset dal file .jsonl caricato nella cartella di lavoro
dataset = load_dataset("json", data_files = {"train": "dataset-limba.jsonl"}, split = "train")

# Definizione del template.
prompt_format = """### Istruzione:
{}

### Risposta:
{}"""

# Recupero il token di fine sequenza dal tokenizer
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    # Ho allineato la chiave a "response" per rispecchiare la struttura del mio file JSONL
    outputs      = examples["response"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # Formatto il testo unendo istruzione e risposta, aggiungendo il token di chiusura
        text = prompt_format.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Applicazione della formattazione alle 1500 istruzioni
dataset = dataset.map(formatting_prompts_func, batched = True)

print(f"Configurazione completata: ho preparato {len(dataset)} esempi per la fase di training.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Configurazione del Trainer per un addestramento profondo (3 Epoche)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        # Aumentiamo le epoche a 3 per fissare i fatti specifici (indirizzi e nomi)
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Inizio l'addestramento potenziato (3 epoche).")

# Avvio del training
trainer_stats = trainer.train()

print("Addestramento terminato.")

In [ ]:
# Test di validazione finale dopo 3 epoche di addestramento.
# Utilizzo il Greedy Decoding per verificare la precisione dei dati anagrafici.

FastLanguageModel.for_inference(model)

domanda_2 = "Chi sei e chi ti ha creato?"
prompt_test = f"### Istruzione:\n{domanda_2}\n\n### Risposta:"

inputs = tokenizer([prompt_test], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    do_sample = False, # Massima precisione
    repetition_penalty = 1.2
)

print("\n--- TEST FINALE (3 EPOCHE): DOMANDA 2 ---")
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Test di validazione finale dopo 3 epoche di addestramento.
# Utilizzo il Greedy Decoding per verificare la precisione dei concetti tecnici.

FastLanguageModel.for_inference(model)

domanda_3 = "Chi è Grazia Deledda?"
prompt_test = f"### Istruzione:\n{domanda_3}\n\n### Risposta:"

inputs = tokenizer([prompt_test], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    do_sample = False, # Massima precisione
    repetition_penalty = 1.2
)

print("\n--- TEST FINALE (3 EPOCHE): DOMANDA 3 ---")
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Test di validazione finale dopo 3 epoche di addestramento.
# Utilizzo il Greedy Decoding per verificare la comprensione dei framework logici.

FastLanguageModel.for_inference(model)

domanda_4 = "Cosa è la fotosintesi?"
prompt_test = f"### Istruzione:\n{domanda_4}\n\n### Risposta:"

inputs = tokenizer([prompt_test], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    do_sample = False, # Massima precisione
    repetition_penalty = 1.2
)

print("\n--- TEST FINALE (3 EPOCHE): DOMANDA 4 ---")
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Esportazione definitiva del modello per l'uso locale
model.save_pretrained_gguf(
    "limba_v1_q4_k_m",
    tokenizer,
    quantization_method = "q4_k_m"
)

In [ ]:
from huggingface_hub import notebook_login

# Authenticate securely without storing the token in the notebook.
# A Hugging Face login dialog will open when this cell is executed.
notebook_login()


In [ ]:
!rm -rf outputs/*